## U-Net

A U-Net is split into an encoder and a decoder, which are mirrored. Layers in the encoder halve spatial dimensions while doubling feature sizes.
With larger receptive fields, increasingly abstract semantic representations of the data can be learned at the cost of sacrificing localization
precision. Layers in the decoder double spatial dimensions while halving feature sizes. This allows the network to localize the semantic
reprsentation at increasingly larger spatial dimensions. Corresponding layers in the encoder are cropped and concatenated to the upsampled layers
in the decoder in order to reintroduce most of the fine spatial information that was preserved in the encoder. Subsequent convolutions learn how
the fine spatial information and the coarse semantic information can be combined effectively. The concatenation also gives layer inputs multiple
shorter paths to reach the output, helping information flow through the network. All of these design decisions result in the encoder learning
*what* is in the data and the decoder learning *where* it is spatially.

Each label corresponds to its own channel, so the final layer in the model uses a 1x1 convolution to map the 64 output channels to the target
number of channels. Softmax is applied per pixel, summing across channels. Cross entropy loss is used as the loss function. Each pixel is
chosen from the channel that coressponds to the pixel's true label.

$E = \sum_{x \in \Omega} w(\mathbf{x}) log(p_{l(x)}(\mathbf(x)))$

Our implementation of the model will differ from the original in several ways.
1. We will use zero-padding so that the output size matches the input size. Border pixels are relatively insignificant, so the slight performance
   decrease from introducing a small amount of extra information should be negligible. Zero padding would also make training and inference much simpler.
2. Ditching the overlap tile strategy in favor of segmenting the entire image, regardless of size. Modern GPUs should be able to hand it.
3. Ditching the weight map and cross entropy loss for Dice loss.
4. Replace each convolution + ReLU with BatchNorm, ReLU, convolution.
5. No elastic deformation, just basic data augmentation.

A few questions of interest:
1. Which normalization scheme (no weight intiialization, basic weight initialization, BatchNorm, GroupNorm, InstanceNorm, etc) would give the best performance?
2. Which learning rate scheduler works best (StepLR, CosineAnnealingLR, ExponentialLR, etc)?

Train U-Net on this dataset: [DRIVE](https://github.com/openmedlab/Awesome-Medical-Dataset/blob/main/resources/DRIVE.md)

Test on this dataset: [Crop/Weed Field Image Dataset](https://datasetninja.com/cwfid#download)

In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import torchvision.transforms.v2.functional as TF

from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import clear_output
from IPython import display

# Map pixel values in the segmentation mask to actual label ids
mapping = torch.full((256,), 255, dtype=torch.long)
mapping[7]  = 0   # road
mapping[8]  = 1   # sidewalk
mapping[11] = 2   # building
mapping[12] = 3   # wall
mapping[13] = 4   # fence
mapping[17] = 5   # pole
mapping[19] = 6   # traffic light
mapping[20] = 7   # traffic sign
mapping[21] = 8   # vegetation
mapping[22] = 9   # terrain
mapping[23] = 10  # sky
mapping[24] = 11  # person
mapping[25] = 12  # rider
mapping[26] = 13  # car
mapping[27] = 14  # truck
mapping[28] = 15  # bus
mapping[31] = 16  # train
mapping[32] = 17  # motorcycle
mapping[33] = 18  # bicycle

def resize(img, target_width):
    aspect_ratio = target_width / img.size[0]
    target_height = int(img.size[1] * aspect_ratio)
    return img.resize((target_width, target_height), Image.Resampling.NEAREST)


class CityscapesDataset(Dataset):
    def __init__(self, img_folder, label_folder, device):
        super().__init__()
        self.imgs = []
        self.label_imgs = []

        for p in Path(img_folder).iterdir():
            if not p.is_file():
                continue

            target_width = 512
            img = resize(Image.open(p), target_width)
            img_tensor = TF.to_image(img).float() / 255
            img.close()

            label_path = f"{label_folder}/{p.stem.split("_leftImg8bit")[0]}_gtFine_labelIds.png"
            segmentation_map = resize(Image.open(label_path), target_width)
            label_tensor = TF.to_image(segmentation_map)[0] # All channels contain the same value
            label_tensor = mapping[label_tensor.long()]
            segmentation_map.close()

            self.imgs.append(img_tensor)
            self.label_imgs.append(label_tensor)


    def __len__(self):
        return len(self.imgs)


    def __getitem__(self, idx):
        return self.imgs[idx], self.label_imgs[idx]


In [ ]:
class UNet(nn.Module):
    def __init__(self, num_labels):
        super().__init__()

        self.layers = nn.ModuleList([])
        in_channels, out_channels, stages = 3, 64, 5

        # Encoder
        for i in range(stages):
            if i != 0:
                self.layers.append(nn.MaxPool2d(2, 2))

            self.layers.extend([
                nn.Conv2d(in_channels, out_channels, 3, stride=1, padding=1),
                nn.InstanceNorm2d(out_channels),
                nn.ReLU(),
                nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1),
                nn.InstanceNorm2d(out_channels),
                nn.ReLU()
            ])

            in_channels = out_channels
            out_channels *= 2

        # Decoder
        for i in range(stages - 1):
            out_channels = in_channels
            in_channels = int(in_channels / 2)

            if i != stages - 1:
                self.layers.append(nn.ConvTranspose2d(out_channels, in_channels, 2, stride=2))

            self.layers.extend([
                nn.Conv2d(out_channels, in_channels, 3, stride=1, padding=1),
                nn.InstanceNorm2d(in_channels),
                nn.ReLU(),
                nn.Conv2d(in_channels, in_channels, 3, stride=1, padding=1),
                nn.InstanceNorm2d(in_channels),
                nn.ReLU()
            ])

        self.layers.append(nn.Conv2d(in_channels, num_labels, 1, stride=1))

    def forward(self, x):
        feature_maps = []
        for layer in self.layers:
            if isinstance(layer, nn.MaxPool2d):
                feature_maps.append(x)
            x = layer(x)
            if isinstance(layer, nn.ConvTranspose2d):
                prev_map = feature_maps.pop()
                prev_map = TF.center_crop(prev_map, output_size=[x.shape[2], x.shape[3]])
                x = torch.cat((prev_map, x), dim=1)
        return x


In [ ]:
def plot_stats(fig, ax, title, errors=None, losses=None):
    clear_output(wait=True)
    ax.clear()

    if losses is not None:
        ax.plot(losses, "b-", label=f"Loss {losses[-1]:.3f}")

    if errors is not None:
        ax.plot(errors, "r-", label=f"Error {errors[-1]:.2f}%")

    ax.autoscale_view(scalex=True, scaley=True)
    ax.set_title(title)
    ax.set_xlabel("Iterations")
    ax.set_ylabel("Stats")
    plt.legend()
    display.display(fig)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

model = UNet(19).to(device)
# TODO: custom loss function (DICE)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=30, gamma=0.1)

losses = []
errors = []
batch_size, epochs = 3, 100
fig, ax = plt.subplots()

img_folder = "/kaggle/input/datasets/electraawais/cityscape-dataset/Cityscape Dataset/leftImg8bit/train/zurich"
label_folder = "/kaggle/input/datasets/electraawais/cityscape-dataset/Fine Annotations/gtFine/train/zurich"
train_samples = CityscapesDataset(img_folder, label_folder, device)
num_train_samples = len(train_samples)
train_loader = DataLoader(dataset=train_samples, batch_size=batch_size, shuffle=True, num_workers=2)

# Train model
print("Started training...")
model.train()
for epoch in range(epochs):
    total_loss = 0

    for i, batch in enumerate(train_loader):
        imgs, labels = batch
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        prediction = model(imgs)
        loss = criterion(prediction, labels)
        total_loss += loss.item()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/{epochs}, Batch: {i+1}/5, LR: {scheduler.get_last_lr()[0]:.3f}")

    losses.append(total_loss / num_train_samples)
    plot_stats(fig, ax, "Training stats", errors=None, losses=losses)
    scheduler.step()
